# Development-only calibrated hybrid comparison

This notebook reads only the existing aggregate `development-hybrid-model-comparison-report.json`. It does not train models, display participant records, or access the locked-test partition. All comparisons use the same nested development-only evaluation.

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import Markdown, display

def repository_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'training').is_dir():
            return candidate
    raise RuntimeError('Repository root not found')

report_path = repository_root() / 'reports' / 'generated' / 'development-hybrid-model-comparison-report.json'
report_bytes = report_path.read_bytes()
report = json.loads(report_bytes.decode('utf-8'))
assert report['scope'] == 'development_only_nested_hybrid_evaluation'
assert report['development_participants'] == 500
assert report['final_test_evaluated'] is False
assert report['privacy']['participant_identifiers_written'] is False
assert report['privacy']['row_level_predictions_written'] is False
assert report['privacy']['report_contains_aggregate_results_only'] is True
pd.DataFrame([{
    'Report dependency': str(report_path.relative_to(repository_root())),
    'SHA-256': hashlib.sha256(report_bytes).hexdigest().upper(),
    'Scope': report['scope'],
    'Development participants': report['development_participants'],
    'Locked test evaluated': report['final_test_evaluated'],
}])

,Report dependency,SHA-256,Scope,Development participants,Locked test evaluated
0,reports\generated\development-hybrid-model-com...,9B1AA013B0BB89382689D784521D13E304E79BA67DF851...,development_only_nested_hybrid_evaluation,500,False


## Primary comparison

The existing best individual model, equal-weight hybrid, and selected-weight hybrid are compared for every domain. Values are means across five repeated outer validation runs. Higher is better except for Brier score.

In [2]:
domain_labels = {'target_communication_concern': 'Communication', 'target_cognitive_concern': 'Cognitive', 'target_social_emotional_concern': 'Social-emotional'}
method_labels = {'logistic_l2_balanced': 'Logistic Regression (individual)', 'xgboost_regularized_balanced': 'XGBoost (individual)', 'equal_weight_soft_voting': 'Equal-weight hybrid', 'selected_weight_soft_voting': 'Selected-weight hybrid'}
metrics_to_show = {'average_precision_pr_auc': 'PR-AUC', 'roc_auc': 'AUROC', 'balanced_accuracy': 'Balanced accuracy', 'sensitivity': 'Sensitivity', 'specificity': 'Specificity', 'accuracy': 'Overall accuracy', 'brier_score': 'Brier score'}
summary_rows = []
for domain_name, domain in report['domains'].items():
    for method_name in (domain['reference_existing_selected_model'], 'equal_weight_soft_voting', 'selected_weight_soft_voting'):
        metrics = domain['methods'][method_name]['repeat_metric_summary']
        row = {'Domain': domain_labels[domain_name], 'Method': method_labels[method_name]}
        row.update({label: metrics[key]['mean'] for key, label in metrics_to_show.items()})
        summary_rows.append(row)
summary_table = pd.DataFrame(summary_rows)
summary_table.style.format({column: '{:.4f}' for column in metrics_to_show.values()}).hide(axis='index')

Domain,Method,PR-AUC,AUROC,Balanced accuracy,Sensitivity,Specificity,Overall accuracy,Brier score
Communication,XGBoost (individual),0.4857,0.8385,0.7225,0.6951,0.7499,0.7432,0.0817
Communication,Equal-weight hybrid,0.4973,0.8610,0.7605,0.8262,0.6948,0.7108,0.0809
Communication,Selected-weight hybrid,0.4951,0.8552,0.7429,0.7902,0.6957,0.7072,0.0811
Cognitive,Logistic Regression (individual),0.5407,0.8849,0.8011,0.8679,0.7342,0.7484,0.0672
Cognitive,Equal-weight hybrid,0.5438,0.8916,0.8220,0.8717,0.7723,0.7828,0.0665
Cognitive,Selected-weight hybrid,0.5477,0.8916,0.8259,0.8755,0.7763,0.7868,0.0664
Social-emotional,XGBoost (individual),0.5812,0.8794,0.7963,0.7627,0.8299,0.8220,0.0703
Social-emotional,Equal-weight hybrid,0.5925,0.8857,0.7947,0.7966,0.7927,0.7932,0.0701
Social-emotional,Selected-weight hybrid,0.5868,0.8844,0.7987,0.7864,0.8109,0.8080,0.0705


## All individual and hybrid results

Both individual members and both hybrid methods are shown. Each metric is the repeated-development-validation mean ± standard deviation.

In [3]:
all_metrics = {'average_precision_pr_auc': 'PR-AUC', 'roc_auc': 'AUROC', 'balanced_accuracy': 'Balanced accuracy', 'sensitivity': 'Sensitivity', 'specificity': 'Specificity', 'accuracy': 'Overall accuracy', 'precision': 'Precision', 'negative_predictive_value': 'NPV', 'f1': 'F1', 'brier_score': 'Brier score'}
detail_rows = []
for domain_name, domain in report['domains'].items():
    for method_name, method in domain['methods'].items():
        metrics = method['repeat_metric_summary']
        row = {'Domain': domain_labels[domain_name], 'Method': method_labels[method_name]}
        row.update({label: f"{metrics[key]['mean']:.4f} ± {metrics[key]['standard_deviation']:.4f}" for key, label in all_metrics.items()})
        detail_rows.append(row)
pd.DataFrame(detail_rows).style.hide(axis='index')

Domain,Method,PR-AUC,AUROC,Balanced accuracy,Sensitivity,Specificity,Overall accuracy,Precision,NPV,F1,Brier score
Communication,Logistic Regression (individual),0.4693 ± 0.0300,0.8649 ± 0.0124,0.7898 ± 0.0138,0.8852 ± 0.0367,0.6943 ± 0.0224,0.7176 ± 0.0171,0.2873 ± 0.0114,0.9777 ± 0.0066,0.4336 ± 0.0133,0.0832 ± 0.0022
Communication,XGBoost (individual),0.4857 ± 0.0167,0.8385 ± 0.0114,0.7225 ± 0.0105,0.6951 ± 0.0395,0.7499 ± 0.0468,0.7432 ± 0.0369,0.2820 ± 0.0299,0.9467 ± 0.0039,0.3997 ± 0.0248,0.0817 ± 0.0017
Communication,Equal-weight hybrid,0.4973 ± 0.0211,0.8610 ± 0.0096,0.7605 ± 0.0101,0.8262 ± 0.0359,0.6948 ± 0.0247,0.7108 ± 0.0181,0.2738 ± 0.0101,0.9666 ± 0.0059,0.4110 ± 0.0099,0.0809 ± 0.0018
Communication,Selected-weight hybrid,0.4951 ± 0.0222,0.8552 ± 0.0086,0.7429 ± 0.0080,0.7902 ± 0.0355,0.6957 ± 0.0355,0.7072 ± 0.0273,0.2662 ± 0.0147,0.9600 ± 0.0049,0.3977 ± 0.0137,0.0811 ± 0.0020
Cognitive,Logistic Regression (individual),0.5407 ± 0.0113,0.8849 ± 0.0033,0.8011 ± 0.0146,0.8679 ± 0.0353,0.7342 ± 0.0163,0.7484 ± 0.0128,0.2794 ± 0.0101,0.9792 ± 0.0052,0.4225 ± 0.0128,0.0672 ± 0.0014
Cognitive,XGBoost (individual),0.5130 ± 0.0298,0.8814 ± 0.0084,0.8153 ± 0.0103,0.8717 ± 0.0338,0.7588 ± 0.0191,0.7708 ± 0.0142,0.3005 ± 0.0106,0.9804 ± 0.0047,0.4466 ± 0.0102,0.0689 ± 0.0025
Cognitive,Equal-weight hybrid,0.5438 ± 0.0242,0.8916 ± 0.0047,0.8220 ± 0.0187,0.8717 ± 0.0470,0.7723 ± 0.0247,0.7828 ± 0.0194,0.3132 ± 0.0191,0.9808 ± 0.0067,0.4603 ± 0.0209,0.0665 ± 0.0019
Cognitive,Selected-weight hybrid,0.5477 ± 0.0293,0.8916 ± 0.0051,0.8259 ± 0.0167,0.8755 ± 0.0492,0.7763 ± 0.0276,0.7868 ± 0.0210,0.3182 ± 0.0189,0.9815 ± 0.0069,0.4661 ± 0.0188,0.0664 ± 0.0020
Social-emotional,Logistic Regression (individual),0.5725 ± 0.0152,0.8770 ± 0.0062,0.7792 ± 0.0130,0.7864 ± 0.0544,0.7719 ± 0.0364,0.7736 ± 0.0264,0.3179 ± 0.0236,0.9646 ± 0.0073,0.4515 ± 0.0172,0.0730 ± 0.0013
Social-emotional,XGBoost (individual),0.5812 ± 0.0299,0.8794 ± 0.0092,0.7963 ± 0.0168,0.7627 ± 0.0208,0.8299 ± 0.0539,0.8220 ± 0.0451,0.3856 ± 0.0647,0.9632 ± 0.0010,0.5088 ± 0.0541,0.0703 ± 0.0026


## Paired confidence intervals

These stratified participant-level paired-bootstrap intervals compare the selected-weight hybrid with the existing selected individual model. Positive PR-AUC differences favour the hybrid; negative Brier differences favour it. They quantify development-sample uncertainty, not external-validation uncertainty.

In [4]:
uncertainty_rows = []
for domain_name, domain in report['domains'].items():
    uncertainty = domain['paired_uncertainty_selected_hybrid_vs_existing_reference']
    ap = uncertainty['differences']['average_precision_pr_auc']; brier = uncertainty['differences']['brier_score']
    ap_ci = ap['confidence_interval_95']; brier_ci = brier['confidence_interval_95']
    uncertainty_rows.append({'Domain': domain_labels[domain_name], 'Reference individual model': method_labels[domain['reference_existing_selected_model']], 'Bootstrap replicates': uncertainty['replicates'], 'PR-AUC difference': ap['mean'], 'PR-AUC difference 95% CI': f"[{ap_ci[0]:.4f}, {ap_ci[1]:.4f}]", 'Brier difference': brier['mean'], 'Brier difference 95% CI': f"[{brier_ci[0]:.4f}, {brier_ci[1]:.4f}]", 'PR-AUC improvement statistically uncertain': ap_ci[0] <= 0 <= ap_ci[1]})
uncertainty_table = pd.DataFrame(uncertainty_rows)
uncertainty_table.style.format({'PR-AUC difference': '{:+.4f}', 'Brier difference': '{:+.4f}'}).hide(axis='index')

Domain,Reference individual model,Bootstrap replicates,PR-AUC difference,PR-AUC difference 95% CI,Brier difference,Brier difference 95% CI,PR-AUC improvement statistically uncertain
Communication,XGBoost (individual),1000,+0.0034,"[-0.0276, 0.0339]",-0.0003,"[-0.0022, 0.0015]",True
Cognitive,Logistic Regression (individual),1000,+0.0009,"[-0.0332, 0.0339]",-0.0005,"[-0.0029, 0.0017]",True
Social-emotional,XGBoost (individual),1000,-0.0004,"[-0.0270, 0.0284]",+0.0004,"[-0.0018, 0.0025]",True


## Weight stability and complementarity

In [5]:
diagnostic_rows = []
for domain_name, domain in report['domains'].items():
    weights = domain['selected_weight_stability']; comp = domain['complementarity']
    diagnostic_rows.append({'Domain': domain_labels[domain_name], 'Mean selected Logistic weight': weights['logistic_weight_mean'], 'Logistic weight SD': weights['logistic_weight_standard_deviation'], 'Weight selection counts': str(weights['selection_counts']), 'Probability correlation': comp['calibrated_probability_pearson_correlation'], 'Decision disagreement rate': comp['decision_disagreement_rate'], 'Double-fault rate': comp['double_fault_rate']})
pd.DataFrame(diagnostic_rows).style.format({'Mean selected Logistic weight': '{:.3f}', 'Logistic weight SD': '{:.3f}', 'Probability correlation': '{:.3f}', 'Decision disagreement rate': '{:.3f}', 'Double-fault rate': '{:.3f}'}).hide(axis='index')

Domain,Mean selected Logistic weight,Logistic weight SD,Weight selection counts,Probability correlation,Decision disagreement rate,Double-fault rate
Communication,0.330,0.119,"{'0.25': 17, '0.5': 8, '0.75': 0}",0.888,0.140,0.198
Cognitive,0.560,0.166,"{'0.25': 3, '0.5': 13, '0.75': 9}",0.906,0.080,0.200
Social-emotional,0.460,0.187,"{'0.25': 9, '0.5': 11, '0.75': 5}",0.923,0.128,0.130


## Scientific conclusion

In [6]:
assert uncertainty_table['PR-AUC improvement statistically uncertain'].all()
display(Markdown("**Conclusion.** The hybrids produce small development-only point-estimate changes relative to the existing selected individual models. However, the paired 95% confidence interval for the selected hybrid’s PR-AUC difference includes zero in all three domains. The calibrated member predictions are strongly correlated and selected weights vary across folds. These results therefore do not establish hybrid superiority and do not justify replacing or freezing the existing individual models. Any hybrid remains a development candidate requiring a new untouched cohort or external validation. No locked-test metric was used in this comparison."))

**Conclusion.** The hybrids produce small development-only point-estimate changes relative to the existing selected individual models. However, the paired 95% confidence interval for the selected hybrid’s PR-AUC difference includes zero in all three domains. The calibrated member predictions are strongly correlated and selected weights vary across folds. These results therefore do not establish hybrid superiority and do not justify replacing or freezing the existing individual models. Any hybrid remains a development candidate requiring a new untouched cohort or external validation. No locked-test metric was used in this comparison.

## Reproducibility boundary

Place the aggregate report at `reports/generated/development-hybrid-model-comparison-report.json` and execute this notebook from anywhere inside the repository. The first output records the exact report SHA-256. The notebook fails closed if the report is not development-only, reports locked-test access, or permits identifiers or row-level predictions.